# **Исследование методов эффективного применения адаптеров (LoRA)  для дообучения моделей в условиях ограниченных вычислительных ресурсов**

# Часть 2
- Эксперимент 2

# Эксперимент 2

План:
- повторяет Эксперимент 1, но с QLoRA (NF4 + 4bit + double quant).

- Сравнивает:
  - q+v
  - q+k+v
  - q+k+v+o

    на r = 4, 8, 16, 32

- Собирает полный лог + таблицу результатов.

In [1]:
!pip install transformers accelerate bitsandbytes peft datasets scikit-learn

import os, random, time
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# Фиксируем seed для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [3]:
# 1. Загрузка и подготовка датасета

dataset = load_dataset("ag_news")
num_labels = 4

# Токенизатор
model_name = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Функция токенизации
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [4]:
# Токенизируем датасет
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Переименовываем label в labels
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Удаляем ненужные колонки
columns_to_remove = ["text"]
tokenized_dataset = tokenized_dataset.remove_columns(columns_to_remove)

# Устанавливаем формат для PyTorch
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

train_dataset = tokenized_dataset["train"]
test_dataset = tokenized_dataset["test"]

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

Train size: 120000, Test size: 7600


In [5]:
# 2. Функция создания модели с QLoRA

def create_qlora_model(rank, target_modules, quant_type="nf4"):
    """
    Создает модель с 4-битным квантованием и LoRA адаптерами

    Args:
        rank: ранг LoRA
        target_modules: список модулей для адаптации
        quant_type: тип квантования ("nf4" или "fp4")
    """

    # Конфигурация квантования
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=quant_type,  # "nf4" или "fp4"
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # Загружаем базовую модель с квантованием
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )

    # Конфигурация LoRA
    lora_config = LoraConfig(
        r=rank,
        lora_alpha=rank * 2,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS
    )

    # Оборачиваем модель в LoRA
    model = get_peft_model(base_model, lora_config)

    return model

In [6]:
# 3. Функция для вычисления метрик

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

In [7]:
# 4. Функция для запуска эксперимента

def run_experiment(rank, target_modules, quant_type="nf4", output_dir="./qlora_outputs"):
    """
    Запускает один эксперимент QLoRA

    Args:
        rank: ранг LoRA
        target_modules: список модулей
        quant_type: тип квантования ("nf4" или "fp4")
        output_dir: директория для сохранения

    Returns:
        dict: метрики эксперимента
    """

    # Создаем уникальное имя для эксперимента
    modules_str = "-".join(target_modules)
    exp_name = f"qlora_{quant_type}_r{rank}_{modules_str}"

    print(f"\n{'='*60}")
    print(f"Запуск: {exp_name}")
    print(f"{'='*60}")


    # Создаем модель
    try:
        model = create_qlora_model(rank, target_modules, quant_type)
        trainable_params = model.num_parameters(only_trainable=True)
        print(f"Trainable parameters: {trainable_params:,}")
    except Exception as e:
        print(f"Ошибка при создании модели: {e}")
        return None

    # Аргументы для тренировки
    training_args = TrainingArguments(
        output_dir=f"{output_dir}/{exp_name}",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="no",
        fp16=False,           # Не используем fp16 с QLoRA
        bf16=True,            # Используем bf16 для стабильности
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=4,
        logging_dir=f"{output_dir}/{exp_name}/logs"
    )

    # Data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # Мониторинг GPU до тренировки
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    start_time = time.time()

    # Тренировка
    trainer.train()

    # Оценка
    metrics = trainer.evaluate()

    end_time = time.time()

    # Сбор метрик GPU
    if torch.cuda.is_available():
        gpu_memory_allocated = torch.cuda.max_memory_allocated() / 1024**3
        gpu_memory_reserved = torch.cuda.max_memory_reserved() / 1024**3
    else:
        gpu_memory_allocated = 0
        gpu_memory_reserved = 0

    # Добавляем метаданные
    metrics["time_sec"] = end_time - start_time
    metrics["rank"] = rank
    metrics["modules"] = modules_str
    metrics["quant_type"] = quant_type
    metrics["gpu_memory_allocated_gb"] = gpu_memory_allocated
    metrics["gpu_memory_reserved_gb"] = gpu_memory_reserved
    metrics["trainable_parameters"] = trainable_params
    metrics["experiment_name"] = exp_name

    # Очистка памяти
    del model
    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\nРезультаты {exp_name}:")
    print(f"  Accuracy: {metrics.get('eval_accuracy', 'N/A'):.4f}")
    print(f"  F1: {metrics.get('eval_f1', 'N/A'):.4f}")
    print(f"  Time: {metrics['time_sec']:.2f} сек")
    print(f"  GPU Memory: {metrics['gpu_memory_allocated_gb']:.2f} GB")

    return metrics

In [9]:
# 5.1. Запуск экспериментов

# Параметры экспериментов
ranks = [4, 8, 16, 32]
target_modules_sets = [
    ["q_proj", "v_proj"]
]
quant_types = ["nf4", "fp4"]  # Тестируем оба типа квантования

# Список для сбора результатов
all_results = []

# Запускаем все комбинации
for quant_type in quant_types:
    for modules in target_modules_sets:
        for rank in ranks:
            print(f"\n>>> Тестируем: {quant_type}, rank={rank}, modules={modules}")

            metrics = run_experiment(
                rank=rank,
                target_modules=modules,
                quant_type=quant_type
            )

            if metrics:
                all_results.append(metrics)

            # Небольшая пауза между экспериментами
            time.sleep(2)


>>> Тестируем: nf4, rank=4, modules=['q_proj', 'v_proj']

Запуск: qlora_nf4_r4_q_proj-v_proj


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 550,912


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.295602,0.301141,0.940921,0.940876


...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 4,364,288


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.283492,0.269374,0.946184,0.946134



Результаты qlora_fp4_r32_q_proj-v_proj:
  Accuracy: 0.9462
  F1: 0.9461
  Time: 6021.37 сек
  GPU Memory: 4.83 GB


In [10]:
# Создаем DataFrame
df_qlora_results = pd.DataFrame(all_results)

# Сохраняем результаты
df_qlora_results.to_csv("qlora_results_1.csv", index=False)
print("\nРезультаты сохранены в qlora_results.csv")


Результаты сохранены в qlora_results.csv


In [14]:
# 5.2.1 Запуск экспериментов

# Параметры экспериментов
ranks = [4] #ranks = [4, 8, 16, 32]
target_modules_sets = [
    ["q_proj", "k_proj", "v_proj"]
]
quant_types = ["nf4", "fp4"]  # Тестируем оба типа квантования

# Список для сбора результатов
all_results_2 = []

# Запускаем все комбинации
for quant_type in quant_types:
    for modules in target_modules_sets:
        for rank in ranks:
            print(f"\n>>> Тестируем: {quant_type}, rank={rank}, modules={modules}")

            metrics = run_experiment(
                rank=rank,
                target_modules=modules,
                quant_type=quant_type
            )

            if metrics:
                all_results_2.append(metrics)

            # Небольшая пауза между экспериментами
            time.sleep(2)


>>> Тестируем: nf4, rank=4, modules=['q_proj', 'k_proj', 'v_proj']

Запуск: qlora_nf4_r4_q_proj-k_proj-v_proj


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 751,616


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.336451,0.291791,0.941184,0.941160


...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 751,616


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.330784,0.289427,0.942105,0.942070



Результаты qlora_fp4_r4_q_proj-k_proj-v_proj:
  Accuracy: 0.9421
  F1: 0.9421
  Time: 6218.00 сек
  GPU Memory: 7.41 GB


In [15]:
# Создаем DataFrame
df_qlora_results_2 = pd.DataFrame(all_results_2)

# Сохраняем результаты
df_qlora_results_2.to_csv("qlora_results_2.csv", index=False)
print("\nРезультаты сохранены в qlora_results_2.csv")


Результаты сохранены в qlora_results_2.csv


In [8]:
# 5.2.2 Запуск экспериментов

# Параметры экспериментов
ranks = [8, 16, 32] #ranks = [4, 8, 16, 32]
target_modules_sets = [
    ["q_proj", "k_proj", "v_proj"]
]
quant_types = ["nf4", "fp4"]  # Тестируем оба типа квантования

# Список для сбора результатов
all_results_2_2 = []

# Запускаем все комбинации
for quant_type in quant_types:
    for modules in target_modules_sets:
        for rank in ranks:
            print(f"\n>>> Тестируем: {quant_type}, rank={rank}, modules={modules}")

            metrics = run_experiment(
                rank=rank,
                target_modules=modules,
                quant_type=quant_type
            )

            if metrics:
                all_results_2_2.append(metrics)

            # Небольшая пауза между экспериментами
            time.sleep(2)


>>> Тестируем: nf4, rank=8, modules=['q_proj', 'k_proj', 'v_proj']

Запуск: qlora_nf4_r8_q_proj-k_proj-v_proj


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 1,497,088


...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 5,969,920


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.307471,0.268459,0.948289,0.948264



Результаты qlora_fp4_r32_q_proj-k_proj-v_proj:
  Accuracy: 0.9483
  F1: 0.9483
  Time: 6197.88 сек
  GPU Memory: 3.88 GB


In [9]:
# Создаем DataFrame
df_qlora_results_2_2 = pd.DataFrame(all_results_2_2)

# Сохраняем результаты
df_qlora_results_2_2.to_csv("qlora_results_2_2.csv", index=False)
print("\nРезультаты сохранены в qlora_results_2_2.csv")


Результаты сохранены в qlora_results_2_2.csv


In [9]:
# 5.3_1. Запуск экспериментов

# Параметры экспериментов
ranks = [4, 8, 16] #ranks = [4, 8, 16, 32]
target_modules_sets = [
    ["q_proj", "k_proj", "v_proj", "o_proj"]
]
quant_types = ["nf4", "fp4"]  # Тестируем оба типа квантования

# Список для сбора результатов
all_results_3_1 = []

# Запускаем все комбинации
for quant_type in quant_types:
    for modules in target_modules_sets:
        for rank in ranks:
            print(f"\n>>> Тестируем: {quant_type}, rank={rank}, modules={modules}")

            metrics = run_experiment(
                rank=rank,
                target_modules=modules,
                quant_type=quant_type
            )

            if metrics:
                all_results_3_1.append(metrics)

            # Небольшая пауза между экспериментами
            time.sleep(2)


>>> Тестируем: nf4, rank=4, modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']

Запуск: qlora_nf4_r4_q_proj-k_proj-v_proj-o_proj


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 1,095,680


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.314725,0.278928,0.947632,0.947622


...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 4,364,288


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.291486,0.260141,0.950921,0.950896



Результаты qlora_fp4_r16_q_proj-k_proj-v_proj-o_proj:
  Accuracy: 0.9509
  F1: 0.9509
  Time: 6398.04 сек
  GPU Memory: 5.09 GB


In [10]:
# Создаем DataFrame
df_qlora_results_3_1 = pd.DataFrame(all_results_3_1)

# Сохраняем результаты
df_qlora_results_3_1.to_csv("qlora_results_3_1.csv", index=False)
print("\nРезультаты сохранены в qlora_results_3_1.csv")


Результаты сохранены в qlora_results_3_1.csv


In [8]:
# 5.3_2. Запуск экспериментов

# Параметры экспериментов
ranks = [32] #ranks = [4, 8, 16, 32]
target_modules_sets = [
    ["q_proj", "k_proj", "v_proj", "o_proj"]
]
quant_types = ["nf4", "fp4"]  # Тестируем оба типа квантования

# Список для сбора результатов
all_results_3_2 = []

# Запускаем все комбинации
for quant_type in quant_types:
    for modules in target_modules_sets:
        for rank in ranks:
            print(f"\n>>> Тестируем: {quant_type}, rank={rank}, modules={modules}")

            metrics = run_experiment(
                rank=rank,
                target_modules=modules,
                quant_type=quant_type
            )

            if metrics:
                all_results_3_2.append(metrics)

            # Небольшая пауза между экспериментами
            time.sleep(2)


>>> Тестируем: nf4, rank=32, modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']

Запуск: qlora_nf4_r32_q_proj-k_proj-v_proj-o_proj


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 8,722,432


...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Trainable parameters: 8,722,432


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.281139,0.252099,0.951053,0.951020



Результаты qlora_fp4_r32_q_proj-k_proj-v_proj-o_proj:
  Accuracy: 0.9511
  F1: 0.9510
  Time: 6379.82 сек
  GPU Memory: 4.03 GB


In [9]:
# Создаем DataFrame
df_qlora_results_3_2 = pd.DataFrame(all_results_3_2)

# Сохраняем результаты
df_qlora_results_3_2.to_csv("qlora_results_3_2.csv", index=False)
print("\nРезультаты сохранены в qlora_results_3_2.csv")


Результаты сохранены в qlora_results_3_2.csv


In [ ]:
# 6. Анализ результатов

# Создаем DataFrame
df_qlora_results = pd.DataFrame(all_results)

# Сохраняем результаты
df_qlora_results.to_csv("qlora_results.csv", index=False)
print("\nРезультаты сохранены в qlora_results.csv")

# Выводим сводную таблицу
print("\n" + "="*80)
print("СВОДКА РЕЗУЛЬТАТОВ QLoRA")
print("="*80)

# Группировка по quant_type и rank
summary = df_qlora_results.groupby(['quant_type', 'rank']).agg({
    'eval_accuracy': ['mean', 'std'],
    'eval_f1': ['mean', 'std'],
    'time_sec': 'mean',
    'gpu_memory_allocated_gb': 'mean'
}).round(4)

print("\nСредние метрики по quant_type и rank:")
print(summary)

# Лучшие конфигурации
print("ТОП-5 ЛУЧШИХ КОНФИГУРАЦИЙ ПО ACCURACY")

top_configs = df_qlora_results.nlargest(5, 'eval_accuracy')[
    ['quant_type', 'rank', 'modules', 'eval_accuracy', 'eval_f1', 'gpu_memory_allocated_gb']
]
print(top_configs.to_string(index=False))

In [ ]:
# 7. Сравнение с обычным LoRA

try:
    # Загружаем результаты обычного LoRA из эксперимента 1
    df_lora_results = pd.read_csv("lora_results.csv")

    # Добавляем столбец типа для сравнения
    df_lora_results['method'] = 'LoRA'
    df_lora_results['quant_type'] = 'none'

    df_qlora_results['method'] = 'QLoRA'

    # Объединяем
    df_comparison = pd.concat([
        df_lora_results[df_lora_results['rank'] > 0],  # исключаем baseline
        df_qlora_results
    ], ignore_index=True)

    # Сравниваем средние метрики по rank
    comparison = df_comparison.groupby(['method', 'rank']).agg({
        'eval_accuracy': 'mean',
        'gpu_memory_allocated_gb': 'mean',
        'time_sec': 'mean'
    }).round(4)

    print("СРАВНЕНИЕ LoRA vs QLoRA (средние по rank)")
    print(comparison)

    # Сохраняем сравнение
    comparison.to_csv("lora_vs_qlora_comparison.csv")

except FileNotFoundError:
    print("\nФайл с результатами LoRA не найден. Пропускаем сравнение.")
except Exception as e:
    print(f"\nОшибка при сравнении: {e}")

print("\nЭксперимент 2 завершен!")